# Learning embeddings

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

# for importing and working with texts
import re
import requests
import string

# pytorch stuff
import torch
from torch.utils.data import Dataset, DataLoader

# 1. Create a data loader to train a model

## Import text and create dictionary

In [2]:
# get raw text from internet
text = requests.get('https://www.gutenberg.org/files/35/35-0.txt').text
# character strings to replace with space
strings2replace = [ '\r\n\r\nâ\x80\x9c','â\x80\x9c','â\x80\x9d','\r\n','â\x80\x94','â\x80\x99','â\x80\x98','_', ]

# use regular expressions (re) to replace those strings with space
for str2match in strings2replace:
    text = re.compile(r'%s'%str2match).sub(' ', text)

# remove non-ASCII characters and numbers and make lower-case
text = re.sub(r'[^\x00-\x7F]+', ' ', text)
text = re.sub('\d+','',text).lower()

# split into words with >1 letters
words = re.split(f'[{string.punctuation}\s]+', text)
words = [item.strip() for item in words if item.strip()]
words = [item for item in words if len(item)>1]

# create the vocabualary
vocab = sorted(set(words))
n_words = len(words)
n_vocab = len(vocab)

# encoder/decoder look-up-tables (as python dictionaries)
word2idx = {word: i for i, word in enumerate(vocab)}
idx2word = {i: word for i, word in enumerate(vocab)}

# show a few keys in the dictionary
print(f'The book contains {n_words:,} words, {n_vocab:,} of which are unique and comprise the vocab.')
print(f'\n\nFirst 10 vocab words:\n',list(word2idx.keys())[:10])

<>:12: SyntaxWarning: invalid escape sequence '\d'
<>:15: SyntaxWarning: invalid escape sequence '\s'
<>:12: SyntaxWarning: invalid escape sequence '\d'
<>:15: SyntaxWarning: invalid escape sequence '\s'
/var/folders/xb/2p09rqt55sj85gtsqtnpdnvh0000gp/T/ipykernel_45906/4043151826.py:12: SyntaxWarning: invalid escape sequence '\d'
  text = re.sub('\d+','',text).lower()
/var/folders/xb/2p09rqt55sj85gtsqtnpdnvh0000gp/T/ipykernel_45906/4043151826.py:15: SyntaxWarning: invalid escape sequence '\s'
  words = re.split(f'[{string.punctuation}\s]+', text)


The book contains 30,698 words, 4,589 of which are unique and comprise the vocab.


First 10 vocab words:
 ['abandon', 'abandoned', 'able', 'abnormally', 'abominable', 'abominations', 'about', 'above', 'abruptly', 'absence']


In [3]:
# parameters for dataset
context_length = 8
stride = 2 # skipping

# initialize
inputs = []
targets = []

# overlapping sequences of context_length
for i in range(0, n_words-context_length, stride):
    # get a few words
    in_seq = words[i: i+context_length]
    target_seq = words[i+1: i+1+context_length]
    # append to the lists
    inputs.append([word2idx[w] for w in in_seq])
    targets.append([word2idx[w] for w in target_seq])

print(inputs[123])
print(targets[123])

[1342, 4304, 4119, 342, 4296, 3388, 1474, 131]
[4304, 4119, 342, 4296, 3388, 1474, 131, 209]


In [4]:
# a closer look:
print("Inputs: ", inputs[4])
print("Targets:", targets[4])
print("")
print("Inputs: ", inputs[5])
print("Targets:", targets[5])
# this is what we need although we need it in torch Dataset/DataLoader format

Inputs:  [2416, 131, 2172, 506, 4451, 783, 2167, 2005]
Targets: [131, 2172, 506, 4451, 783, 2167, 2005, 4042]

Inputs:  [2172, 506, 4451, 783, 2167, 2005, 4042, 2416]
Targets: [506, 4451, 783, 2167, 2005, 4042, 2416, 2006]


In [5]:
# we need each list to be a tensor
torch.tensor(inputs[4])

tensor([2416,  131, 2172,  506, 4451,  783, 2167, 2005])

## Create a class for a dataset object

In [6]:
# create a class for a dataset
class WordDataset(Dataset):
    def __init__(self, text, word2idx, context_length=8, stride=4) -> None:
        # initialize
        self.inputs = []
        self.targets = []
        self.word2idx = word2idx # stored locally in the object

        # overlapping sequences of context_length
        for i in range(0, len(text)-context_length, stride):
            # get a few words
            in_seq = text[i: i+context_length]
            targ_seq = text[i+1: i+1+context_length]

            # append to the lists
            self.inputs.append(torch.tensor([word2idx[w] for w in in_seq]))
            self.targets.append(torch.tensor([word2idx[w] for w in targ_seq]))
    
    def __len__(self):
        return len(self.inputs)
    
    def __getitem__(self, index):
        return self.inputs[index], self.targets[index]

# create an instance!
context_length = 6
stride = 3 # skipping over tokens
text_dataset = WordDataset(words, word2idx, context_length, stride)

text_dataset[4]

(tensor([4451,  783, 2167, 2005, 4042, 2416]),
 tensor([ 783, 2167, 2005, 4042, 2416, 2006]))

## a dataloader for training

In [7]:
# also need a dataloader
dataloader = DataLoader(
    text_dataset,
    batch_size=2, # 2 for looking; 32 for training
    shuffle=True,
    drop_last=True
)

# lets have a look at the indices
X, y = next(iter(dataloader))
print("Inputs:")
print(X), print("")

print("Targets:")
print(y), print("\n\n\n")

# and the words
print("Inputs in words (first batch):")
print([idx2word[item.item()] for item in X[0]])
print("")

print("Targets in words (first batch):")
print([idx2word[item.item()] for item in y[0]])

Inputs:
tensor([[4188, 4119, 1608, 3131,    6, 2186],
        [ 919, 2731, 4429, 1349, 1606, 4050]])

Targets:
tensor([[4119, 1608, 3131,    6, 2186, 2041],
        [2731, 4429, 1349, 1606, 4050,  137]])




Inputs in words (first batch):
['tried', 'to', 'frame', 'question', 'about', 'it']

Targets in words (first batch):
['to', 'frame', 'question', 'about', 'it', 'in']
